# BlueCarbon-AI: kelp run (Baja California to Washington, 1984 to today)

Runs the whole kelp product and writes `results_kelp.zip` for the website:
- **Sentinel-2 canopy every quarter since 2016** for 11 kelp beds (4 new: Isla Todos Santos and Isla Natividad in Baja California, Orford Reef in Oregon, Neah Bay in Washington).
- **Our own Landsat canopy record, 1984 to 2015**, with the same method.
- **Calibration per zone** (giant kelp and bull kelp separately), and **every bed scored on data it was not tuned on**.
- **Sea surface temperature** every quarter since 1984 (NOAA OISST), and a **summer satellite picture** for every year.

No GPU needed: pick a normal (CPU) runtime; it can run in its own tab while the main training runs. The first run takes several hours (about 1,400 small Landsat downloads); everything is saved to Drive as it goes, so if it stops, just Run all again and it continues.

**Click Runtime → Run all.** It pauses for the Earth Engine and Drive sign-ins. At the end `results_kelp.zip` downloads and is also saved to My Drive → bluecarbon-ai-results.

In [ ]:
%cd /content
!git clone -q https://github.com/yanicksanchez14-creator/bluecarbon-ai.git 2>/dev/null; git -C /content/bluecarbon-ai fetch -q origin && git -C /content/bluecarbon-ai reset -q --hard origin/main
%cd /content/bluecarbon-ai
!git log -1 --format="Code version: %h  %s"
!pip -q install -e ".[gee]"

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project="bluecarbon-ai")
print("Earth Engine OK:", ee.Number(1).getInfo() == 1)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
cache = "/content/drive/MyDrive/bluecarbon-ai-cache/sites"
link = "/content/bluecarbon-ai/runs/default/sites"
os.makedirs(os.path.dirname(link), exist_ok=True)
if not os.path.islink(link):
    os.symlink(cache, link)
k = os.path.join(cache, "_kelp")
print("Kelp beds with saved quarters:", sorted(d for d in os.listdir(k) if os.path.isdir(os.path.join(k, d))))

In [ ]:
!pip -q install xarray netCDF4
!python scripts/kelp_pipeline.py

import os, shutil, datetime
from google.colab import files
if os.path.exists("results_kelp.zip"):
    dst = "/content/drive/MyDrive/bluecarbon-ai-results"
    os.makedirs(dst, exist_ok=True)
    shutil.copy("results_kelp.zip", os.path.join(dst, f"results_kelp_{datetime.date.today():%Y%m%d}.zip"))
    files.download("results_kelp.zip")
else:
    print("No results_kelp.zip: see the error above, or run the cells again to resume.")